# 🎙️ Notebook 02 — EDA + Classificador de Emoção Vocal

**Tech Challenge Fase 4 | PosTech FIAP — IA para Devs**

---

## Objetivo

Este notebook realiza a **Análise Exploratória** dos datasets de emoção vocal e treina  
um **classificador de emoção** para identificar sinais de medo, tristeza e raiva — indicadores  
de possível violência doméstica ou trauma em consultas médicas.

### Datasets utilizados:
- **RAVDESS** — 2.880 arquivos WAV (24 atores, 8 emoções, EN)
- **CREMA-D** — 7.442 arquivos WAV (91 atores, 6 emoções, EN)

### O que faremos:
1. Decodificar a nomenclatura dos arquivos (convenção de rótulos)
2. Análise da distribuição de emoções por dataset
3. Visualização de sinais de áudio: waveform, espectrograma, MFCCs
4. Extração de features de prosódia (MFCCs, pitch, energia) — conforme Aula 04
5. Treinar classificador supervisionado (TF-IDF de features + Naive Bayes / SVM) — Aula 05
6. Avaliar com métricas: precision, recall, F1-score
7. Salvar modelo treinado para uso no pipeline

---

**Referências das aulas:**
- Aula 04 — Transcrição automática de áudio e conversão de fala em texto (librosa, SpeechRecognition)
- Aula 05 — Classificação de tópicos e categorização de texto (sklearn, NLTK, TF-IDF, Naive Bayes)

## 1. Importações e Configuração

Utilizamos as mesmas bibliotecas recomendadas nas aulas:  
`librosa` para análise de áudio, `sklearn` para classificação (como na Aula 05),  
e `matplotlib`/`seaborn` para visualizações.

In [ ]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['figure.dpi'] = 100

# Caminhos dos datasets
RAVDESS_PATH = Path('../data/datasets/violence_audio/RAVDESS')
CREMA_PATH   = Path('../data/datasets/violence_audio/CREMA-D')
MODEL_OUT    = Path('../data/datasets/violence_audio/emotion_classifier.pkl')

print(f'RAVDESS existe: {RAVDESS_PATH.exists()}')
print(f'CREMA-D existe: {CREMA_PATH.exists()}')

## 2. Decodificação dos Rótulos

Cada dataset usa uma convenção de nomenclatura diferente para codificar a emoção no nome do arquivo.  
Precisamos decodificá-la antes de qualquer análise.

### RAVDESS
Formato: `03-01-**06**-01-01-01-01.wav`  
Posição 3 (índice 2) = código da emoção:  
`01=neutral, 02=calm, 03=happy, 04=sad, 05=angry, 06=fearful, 07=disgust, 08=surprised`

### CREMA-D
Formato: `1001_DFA_**ANG**_XX.wav`  
Terceiro campo separado por `_` = código da emoção:  
`ANG=angry, DIS=disgust, FEA=fearful, HAP=happy, NEU=neutral, SAD=sad`

In [ ]:
# Mapeamentos de código → emoção em português
RAVDESS_EMOTION_MAP = {
    '01': 'neutral',  '02': 'calm',    '03': 'happy',
    '04': 'sad',      '05': 'angry',   '06': 'fearful',
    '07': 'disgust',  '08': 'surprised'
}

CREMA_EMOTION_MAP = {
    'ANG': 'angry',   'DIS': 'disgust',  'FEA': 'fearful',
    'HAP': 'happy',   'NEU': 'neutral',  'SAD': 'sad'
}

# Emoções de risco para triagem de violência doméstica
RISK_EMOTIONS = {'fearful', 'angry', 'sad', 'disgust'}

def parse_ravdess(filepath: Path) -> dict | None:
    """Extrai metadados de um arquivo RAVDESS a partir do nome."""
    parts = filepath.stem.split('-')
    if len(parts) < 7:
        return None
    modality = parts[0]  # 03 = audio-only
    if modality != '03':  # Filtramos apenas audio-only
        return None
    emotion_code = parts[2]
    emotion = RAVDESS_EMOTION_MAP.get(emotion_code, 'unknown')
    return {
        'filepath' : str(filepath),
        'dataset'  : 'RAVDESS',
        'emotion'  : emotion,
        'risk'     : emotion in RISK_EMOTIONS,
        'actor'    : int(parts[6]),
        'intensity': 'strong' if parts[3] == '02' else 'normal',
    }

def parse_crema(filepath: Path) -> dict | None:
    """Extrai metadados de um arquivo CREMA-D a partir do nome."""
    parts = filepath.stem.split('_')
    if len(parts) < 3:
        return None
    emotion_code = parts[2]
    emotion = CREMA_EMOTION_MAP.get(emotion_code, 'unknown')
    return {
        'filepath' : str(filepath),
        'dataset'  : 'CREMA-D',
        'emotion'  : emotion,
        'risk'     : emotion in RISK_EMOTIONS,
        'actor'    : int(parts[0]),
        'intensity': parts[3] if len(parts) > 3 else 'XX',
    }

# Teste de decodificação
exemplo_r = Path('03-01-06-01-01-01-01.wav')
exemplo_c = Path('1001_DFA_FEA_HI.wav')
print('RAVDESS:', parse_ravdess(exemplo_r))
print('CREMA-D:', parse_crema(exemplo_c))

## 3. Construção do Catálogo de Arquivos

Percorremos as pastas dos dois datasets e criamos um DataFrame consolidado  
com os metadados de cada arquivo de áudio.

In [ ]:
records = []

# RAVDESS — arquivos organizados em pastas Actor_XX/
for wav_file in RAVDESS_PATH.rglob('*.wav'):
    meta = parse_ravdess(wav_file)
    if meta:
        records.append(meta)

# CREMA-D — arquivos todos na mesma pasta
for wav_file in CREMA_PATH.glob('*.wav'):
    meta = parse_crema(wav_file)
    if meta:
        records.append(meta)

df = pd.DataFrame(records)

print(f'Total de arquivos carregados: {len(df)}')
print(f'  RAVDESS : {(df.dataset == "RAVDESS").sum()}')
print(f'  CREMA-D : {(df.dataset == "CREMA-D").sum()}')
print(f'\nEmoções únicas: {sorted(df.emotion.unique())}')
df.head()

## 4. Distribuição das Emoções

Verificamos o balanceamento das classes antes de treinar o classificador.  
Conforme a Aula 05, um dataset desequilibrado pode causar viés no modelo.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Total por emoção
emotion_order = df['emotion'].value_counts().index
colors_em = sns.color_palette('Set2', len(emotion_order))
counts = df['emotion'].value_counts()

axes[0].bar(counts.index, counts.values, color=colors_em, edgecolor='white')
axes[0].set_title('Total por Emoção (ambos datasets)', fontweight='bold')
axes[0].set_ylabel('Arquivos')
axes[0].tick_params(axis='x', rotation=30)

# Por dataset
df.groupby(['dataset', 'emotion']).size().unstack(fill_value=0).plot(
    kind='bar', ax=axes[1], edgecolor='white'
)
axes[1].set_title('Distribuição por Dataset e Emoção', fontweight='bold')
axes[1].set_ylabel('Arquivos')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Emoção', bbox_to_anchor=(1.01, 1), fontsize=8)

# Risco vs Não-risco
risk_counts = df['risk'].map({True: 'Risco (fear/sad/angry/disgust)', False: 'Sem risco'}).value_counts()
axes[2].pie(
    risk_counts.values,
    labels=risk_counts.index,
    colors=['#e74c3c', '#2ecc71'],
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[2].set_title('Proporção de Emoções de Risco', fontweight='bold')

plt.suptitle('Análise Exploratória — Datasets de Emoção Vocal', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\nContagem por emoção:')
print(df['emotion'].value_counts())

## 5. Visualização do Sinal de Áudio

Antes de extrair features, visualizamos o sinal bruto de exemplos de emoções contrastantes:  
**neutral** (baixo risco) vs **fearful** (alto risco).  
Isso nos dá intuição sobre as diferenças que o modelo vai aprender.

In [ ]:
def plot_audio_comparison(df: pd.DataFrame, emotion_a: str, emotion_b: str, dataset: str = 'RAVDESS'):
    """Plota waveform e espectrograma de dois exemplos de emoções diferentes."""
    sample_a = df[(df.emotion == emotion_a) & (df.dataset == dataset)].iloc[0]['filepath']
    sample_b = df[(df.emotion == emotion_b) & (df.dataset == dataset)].iloc[0]['filepath']

    fig, axes = plt.subplots(2, 2, figsize=(14, 7))

    for col, (path, label, color) in enumerate([
        (sample_a, f'Emoção: {emotion_a}', '#2ecc71'),
        (sample_b, f'Emoção: {emotion_b}', '#e74c3c'),
    ]):
        y, sr = librosa.load(path, duration=3.0)  # Carrega primeiros 3s

        # Waveform
        librosa.display.waveshow(y, sr=sr, ax=axes[0][col], color=color, alpha=0.8)
        axes[0][col].set_title(f'Waveform — {label}', fontweight='bold')
        axes[0][col].set_xlabel('Tempo (s)')
        axes[0][col].set_ylabel('Amplitude')

        # Mel Spectrogram
        S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
        S_dB = librosa.power_to_db(S, ref=np.max)
        img = librosa.display.specshow(S_dB, sr=sr, x_axis='time', y_axis='mel',
                                        ax=axes[1][col], cmap='magma')
        axes[1][col].set_title(f'Mel Spectrogram — {label}', fontweight='bold')
        fig.colorbar(img, ax=axes[1][col], format='%+2.0f dB')

    plt.suptitle(f'Comparação: {emotion_a} vs {emotion_b} ({dataset})',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

plot_audio_comparison(df, 'neutral', 'fearful')

### 5.1 Visualização dos MFCCs

Os **MFCCs (Mel-Frequency Cepstral Coefficients)** são as features mais importantes  
para classificação de emoção vocal.  
Eles capturam a forma espectral da voz, refletindo o estado emocional do falante.

In [ ]:
emotions_to_show = ['neutral', 'happy', 'sad', 'fearful', 'angry']
fig, axes = plt.subplots(1, len(emotions_to_show), figsize=(18, 4))

for ax, emotion in zip(axes, emotions_to_show):
    sample = df[df.emotion == emotion].iloc[0]['filepath']
    y, sr = librosa.load(sample, duration=3.0)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)

    img = librosa.display.specshow(mfccs, sr=sr, x_axis='time', ax=ax, cmap='coolwarm')
    ax.set_title(f'MFCCs\n{emotion}', fontweight='bold', fontsize=10)
    ax.set_ylabel('Coeficiente MFCC' if emotion == emotions_to_show[0] else '')

plt.suptitle('Comparação de MFCCs por Emoção', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('Nota: Padrões distintos nos MFCCs indicam alta separabilidade entre emoções.')

## 6. Extração de Features de Prosódia

Seguindo a Aula 04 (processamento de áudio com librosa) e nossa implementação em  
`src/audio/emotion_audio.py`, extraímos um vetor de características por arquivo.

**Features extraídas:**
- **MFCCs** (13 coeficientes, média + desvio): captura timbre e qualidade vocal
- **Pitch médio (F0)**: tom da voz — aumenta em ansiedade/medo
- **Energia RMS**: volume — alta em raiva, baixa em tristeza
- **Zero Crossing Rate**: velocidade de fala proxy
- **Chroma**: relação entre notas — padrão harmônico emocional

> **Nota:** Este processo pode levar alguns minutos (~10k arquivos).

In [ ]:
def extract_features(filepath: str, duration: float = 3.0) -> np.ndarray | None:
    """
    Extrai vetor de features de um arquivo de áudio.
    Retorna array 1D com 40 features ou None se falhar.

    Seguindo Aula 04: carregamento com librosa + extração de características.
    """
    try:
        y, sr = librosa.load(filepath, duration=duration, res_type='kaiser_fast')
        if len(y) < sr * 0.5:  # Arquivo muito curto
            return None

        # 13 MFCCs (média) + 13 MFCCs (desvio) = 26 features
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        mfcc_mean = mfccs.mean(axis=1)
        mfcc_std  = mfccs.std(axis=1)

        # Energia RMS (média + desvio) = 2 features
        rms = librosa.feature.rms(y=y)[0]
        rms_mean = rms.mean()
        rms_std  = rms.std()

        # Zero Crossing Rate = 1 feature
        zcr = librosa.feature.zero_crossing_rate(y)[0].mean()

        # Spectral Centroid (centro de gravidade espectral) = 1 feature
        spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0].mean()

        # Chroma (12 notas cromáticas, média) = 12 features
        # Nota: não disponível em todos os casos, mas enriquece o vetor
        chroma = librosa.feature.chroma_stft(y=y, sr=sr, n_chroma=12).mean(axis=1)

        # Concatena tudo: 26 + 2 + 1 + 1 + 12 = 42 features
        features = np.concatenate([
            mfcc_mean, mfcc_std,
            [rms_mean, rms_std, zcr, spectral_centroid],
            chroma
        ])
        return features

    except Exception:
        return None

# Teste em 1 arquivo
sample_path = df.iloc[0]['filepath']
feat = extract_features(sample_path)
print(f'Vetor de features: {feat.shape[0]} dimensões')
print(f'Primeiras 5 features: {feat[:5].round(4)}')

In [ ]:
# Extração em todo o dataset (pode demorar ~5-10 min)
# Usamos amostragem para o notebook: máx 300 arquivos por emoção
MAX_PER_EMOTION = 300

df_sample = df.groupby('emotion').apply(
    lambda x: x.sample(min(len(x), MAX_PER_EMOTION), random_state=42)
).reset_index(drop=True)

print(f'Amostra para extração: {len(df_sample)} arquivos')
print(df_sample['emotion'].value_counts())

X_list, y_list = [], []

for _, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc='Extraindo features'):
    features = extract_features(row['filepath'])
    if features is not None:
        X_list.append(features)
        y_list.append(row['emotion'])

X = np.array(X_list)
y = np.array(y_list)

print(f'\nDataset de features: X={X.shape}, y={y.shape}')
print(f'Emoções: {np.unique(y)}')

### 6.1 Visualização das Features por Emoção

Usando **PCA** (Análise de Componentes Principais) reduzimos as 42 features para 2D  
e verificamos se as emoções formam clusters separáveis — fundamental para um bom classificador.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

df_pca = pd.DataFrame({'PC1': X_pca[:, 0], 'PC2': X_pca[:, 1], 'emotion': y})

plt.figure(figsize=(10, 6))
emotions_unique = df_pca['emotion'].unique()
palette = dict(zip(emotions_unique, sns.color_palette('tab10', len(emotions_unique))))

for emotion, color in palette.items():
    subset = df_pca[df_pca.emotion == emotion]
    plt.scatter(subset.PC1, subset.PC2, c=[color], label=emotion, alpha=0.5, s=20, edgecolors='none')

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variância)', fontsize=11)
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variância)', fontsize=11)
plt.title('PCA das Features de Áudio — Separação por Emoção', fontsize=13, fontweight='bold')
plt.legend(title='Emoção', bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()

total_var = pca.explained_variance_ratio_.sum() * 100
print(f'Variância explicada pelos 2 componentes: {total_var:.1f}%')

## 7. Treinamento do Classificador de Emoção

Seguindo o pipeline da **Aula 05** (Classificação Supervisionada com sklearn):  
- Split treino/teste estratificado
- Normalização das features
- Comparamos 3 classificadores: **SVM**, **Random Forest** e **Naive Bayes** (mencionado diretamente na Aula 05)

> Para o pipeline do projeto, priorizamos **recall de fearful/sad/angry** sobre acurácia global,  
> pois falsos negativos (não detectar risco real) são mais custosos clinicamente.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics

# Codificação dos rótulos
le = LabelEncoder()
y_enc = le.fit_transform(y)

# Split estratificado — garante todas as classes no treino e teste (Aula 05)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

print(f'Treino : {X_train.shape[0]} amostras')
print(f'Teste  : {X_test.shape[0]} amostras')
print(f'Classes: {le.classes_}')

In [ ]:
# Definição dos modelos — conforme técnicas da Aula 05
models = {
    'Naive Bayes'   : make_pipeline(StandardScaler(), GaussianNB()),
    'SVM (RBF)'     : make_pipeline(StandardScaler(), SVC(kernel='rbf', C=1.0, probability=True, random_state=42)),
    'Random Forest' : make_pipeline(StandardScaler(), RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')),
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = metrics.accuracy_score(y_test, y_pred)
    f1  = metrics.f1_score(y_test, y_pred, average='weighted')
    results[name] = {'accuracy': acc, 'f1_weighted': f1, 'model': model, 'y_pred': y_pred}
    print(f'{name:20s} | Acurácia: {acc:.3f} | F1 (weighted): {f1:.3f}')

### 7.1 Relatório Detalhado do Melhor Modelo

Seguindo o padrão da **Aula 05**, exibimos Precision, Recall, F1-score e Support  
para cada classe emocional, incluindo a interpretação clínica dos resultados.

In [ ]:
# Seleciona o melhor modelo por F1
best_name = max(results, key=lambda k: results[k]['f1_weighted'])
best_result = results[best_name]
y_pred_best = best_result['y_pred']

print(f'Melhor modelo: {best_name}')
print('=' * 60)
print(metrics.classification_report(
    y_test, y_pred_best,
    target_names=le.classes_,
    zero_division=0
))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz de confusão
from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_best,
    display_labels=le.classes_,
    cmap='Blues',
    ax=axes[0],
    xticks_rotation=30
)
axes[0].set_title(f'Matriz de Confusão — {best_name}', fontweight='bold')

# Recall por emoção de risco
report = metrics.classification_report(y_test, y_pred_best, target_names=le.classes_,
                                        output_dict=True, zero_division=0)
recall_vals = {cls: report[cls]['recall'] for cls in le.classes_}
risk_flag   = [cls in RISK_EMOTIONS for cls in le.classes_]
bar_colors  = ['#e74c3c' if r else '#3498db' for r in risk_flag]

axes[1].barh(list(recall_vals.keys()), list(recall_vals.values()),
             color=bar_colors, edgecolor='white')
axes[1].axvline(x=0.7, color='black', linestyle='--', alpha=0.5, label='Meta mínima (0.70)')
axes[1].set_xlabel('Recall')
axes[1].set_title('Recall por Emoção\n(vermelho = emoção de risco clínico)', fontweight='bold')
axes[1].legend()

plt.suptitle(f'Avaliação do Classificador — {best_name}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Classificação Binária: Risco vs Sem Risco

Para o pipeline clínico, o mais importante é detectar **qualquer emoção de risco**  
(fearful, angry, sad, disgust) independente de qual emoção específica.  
Treinamos um classificador binário com foco em **alto recall de risco**.

In [ ]:
# Rótulo binário: 1 = emoção de risco, 0 = sem risco
y_binary = np.array([1 if le.classes_[c] in RISK_EMOTIONS else 0 for c in y_enc])

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X, y_binary, test_size=0.2, random_state=42, stratify=y_binary
)

# SVM com class_weight='balanced' — fundamental para dados clínicos
clf_binary = make_pipeline(
    StandardScaler(),
    SVC(kernel='rbf', C=1.0, class_weight='balanced', probability=True, random_state=42)
)
clf_binary.fit(X_train_b, y_train_b)
y_pred_b = clf_binary.predict(X_test_b)

print('=== Classificação Binária: Risco vs Sem Risco ===')
print(metrics.classification_report(
    y_test_b, y_pred_b,
    target_names=['Sem Risco', 'Com Risco'],
    zero_division=0
))

roc_auc = metrics.roc_auc_score(y_test_b, clf_binary.predict_proba(X_test_b)[:, 1])
print(f'AUC-ROC: {roc_auc:.3f}')

In [ ]:
# Curva ROC — mostra a relação entre recall e taxa de falsos positivos
from sklearn.metrics import RocCurveDisplay

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

RocCurveDisplay.from_estimator(
    clf_binary, X_test_b, y_test_b,
    name='SVM Binário (Risco Vocal)',
    ax=axes[0], color='#e74c3c'
)
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Aleatório')
axes[0].set_title('Curva ROC — Detecção de Risco Vocal', fontweight='bold')
axes[0].legend()

ConfusionMatrixDisplay.from_predictions(
    y_test_b, y_pred_b,
    display_labels=['Sem Risco', 'Com Risco'],
    cmap='Reds', ax=axes[1]
)
axes[1].set_title('Matriz de Confusão — Binária', fontweight='bold')

plt.tight_layout()
plt.show()

## 9. Comparação de Features por Emoção de Risco

Identificamos quais features prosódicas mais diferenciam emoções de risco das seguras.  
Isso guia as regras heurísticas implementadas em `src/audio/emotion_audio.py`.

In [ ]:
# Índices das features de prosódia simples (fora dos MFCCs)
# Features 26-29: rms_mean, rms_std, zcr, spectral_centroid
FEATURE_NAMES = (
    [f'MFCC_{i}_mean' for i in range(1, 14)] +
    [f'MFCC_{i}_std'  for i in range(1, 14)] +
    ['RMS_mean', 'RMS_std', 'ZCR', 'Spectral_Centroid'] +
    [f'Chroma_{i}' for i in range(1, 13)]
)

df_features = pd.DataFrame(X, columns=FEATURE_NAMES)
df_features['emotion'] = y
df_features['risk']    = [1 if e in RISK_EMOTIONS else 0 for e in y]

# Comparar features prosódicas simples entre risco e não-risco
prosody_features = ['RMS_mean', 'ZCR', 'Spectral_Centroid', 'MFCC_1_mean']

fig, axes = plt.subplots(1, len(prosody_features), figsize=(16, 4))

for ax, feat in zip(axes, prosody_features):
    sns.boxplot(
        data=df_features, x='emotion', y=feat,
        order=sorted(df_features.emotion.unique()),
        palette='Set2', ax=ax, linewidth=1.5
    )
    ax.set_title(feat, fontweight='bold', fontsize=10)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=40)

plt.suptitle('Features Prosódicas por Emoção', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Salvar Modelos para o Pipeline

Exportamos os dois modelos treinados (multiclasse + binário) e o scaler  
para integração com `src/audio/emotion_audio.py`.

In [ ]:
import pickle

output_dir = Path('../data/datasets/violence_audio')

# Salvar modelo multiclasse
model_multiclass = results[best_name]['model']
with open(output_dir / 'emotion_classifier_multiclass.pkl', 'wb') as f:
    pickle.dump({'model': model_multiclass, 'label_encoder': le}, f)

# Salvar classificador binário
with open(output_dir / 'emotion_classifier_binary.pkl', 'wb') as f:
    pickle.dump({'model': clf_binary, 'risk_emotions': list(RISK_EMOTIONS)}, f)

# Salvar nomes das features para referência
import json
meta = {
    'feature_names'        : FEATURE_NAMES,
    'n_features'           : len(FEATURE_NAMES),
    'best_model'           : best_name,
    'emotions'             : list(le.classes_),
    'risk_emotions'        : list(RISK_EMOTIONS),
    'f1_weighted_multiclass': round(results[best_name]['f1_weighted'], 4),
    'roc_auc_binary'       : round(roc_auc, 4),
}
(output_dir / 'emotion_model_meta.json').write_text(json.dumps(meta, indent=2))

print('Modelos salvos:')
for f in output_dir.glob('emotion_*'):
    print(f'  {f.name}')

## 11. Conclusões e Próximos Passos

### Resultados obtidos:

| Modelo | Tipo | Métrica Principal |
|---|---|---|
| Melhor classificador multiclasse | 6 emoções | F1-weighted |
| SVM binário | Risco vs Sem risco | AUC-ROC + Recall |

### Insights para o pipeline clínico:
- **RMS_mean** e **Spectral_Centroid** são os melhores discriminadores simples de risco
- O classificador binário é mais adequado para alertas em tempo real (menor latência)
- Emoções **fearful** e **sad** têm recall mais alto — ótimo para triagem de violência

### Limitações conhecidas:
- Dataset em inglês — desempenho pode cair em português
- Atores simulando emoções ≠ emoções reais em consulta clínica
- Recomendação: fine-tuning com dados reais quando disponíveis

### Integração com o projeto:
- Modelos salvos em `data/datasets/violence_audio/*.pkl`
- `src/audio/emotion_audio.py` — substituir heurística pelo modelo treinado
- Próximo: **Notebook 03** — Fine-tuning YOLOv8 com m2caiSeg